In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)
import os.path as osp
from torch_geometric.loader import DataLoader
import torch
from torch_geometric.data import Dataset, download_url
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.nn import GATConv
import optuna

import pickle
# !pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
# !pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
# !pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:
torch.cuda.is_available()

In [ ]:
class MyOwnDataset(Dataset):
    def __init__(self, root, transform=None, pre_transform=None, pre_filter=None):
        super().__init__(root, transform, pre_transform, pre_filter)

    @property
    def raw_file_names(self):
        return ""

    @property
    def processed_file_names(self):
        files = os.listdir(self.processed_dir)
        filtered_files = [file for file in files if file.endswith(".pt")]
        return filtered_files

    def download(self):
        # Download to `self.raw_dir`.
        pass

    def process(self):
       
        for raw_path in self.raw_paths:
            # Read data from `raw_path`.
            data = Data(...)

            if self.pre_filter is not None and not self.pre_filter(data):
                continue

            if self.pre_transform is not None:
                data = self.pre_transform(data)

            torch.save(data, osp.join(self.processed_dir, f'data_{idx}.pt'))
            idx += 1

    def len(self):
        return len(self.processed_file_names)

    def get(self, idx):
        data = torch.load(osp.join(self.processed_dir, f'data_{idx}.pt'))
        return data

In [ ]:
dataset= MyOwnDataset(root='fakeddit_only_train')

In [ ]:
import json

with open("data_mapping_only_train.json","r") as f:
     data_mapping = json.load(f)  
    
mapping_id = {v: k for k, v in data_mapping.items()}

In [ ]:
with open("train_ids.pkl","rb") as f:
     train_ids = pickle.load(f)
with open("valid_ids.pkl","rb") as f:
     valid_ids = pickle.load(f)
with open("test_ids.pkl","rb") as f:
     test_ids = pickle.load(f)

In [ ]:
## TIMESTAMP BASED DATA SPLIT
import random
# Fix the seed for reproducibility
random.seed(42)

random.shuffle(train_ids)
random.shuffle(valid_ids)
random.shuffle(test_ids)
final_test_ids = [i for i in test_ids if i in data_mapping]
train_dataset = dataset[tuple([data_mapping[i] for i in train_ids if i in data_mapping])]
test_dataset = dataset[tuple([data_mapping[i] for i in test_ids if i in data_mapping])]
valid_dataset = dataset[tuple([data_mapping[i] for i in valid_ids if i in data_mapping])]

print(f'Number of training graphs: {len(train_dataset)}')
print(f'Number of Validation graphs: {len(valid_dataset)}')
print(f'Number of test graphs: {len(test_dataset)}')

In [ ]:
print(len(final_test_ids))

In [ ]:
# ## Using this - speeds up the computing
data_list_Train =[]
for i,data in enumerate(train_dataset):
    data_list_Train.append(data)
    if i%10000==0:
        print(i)

data_list_Valid =[]
for i,data in enumerate(valid_dataset):
    data_list_Valid.append(data)
    if i%10000==0:
        print(i)
        
data_list_Test =[]
for i,data in enumerate(test_dataset):
    data_list_Test.append(data)
    if i%10000==0:
        print(i)
    

In [ ]:
train_loader = DataLoader(data_list_Train, batch_size=256, shuffle=True)
valid_loader = DataLoader(data_list_Valid, batch_size=256, shuffle=True)
test_loader = DataLoader(data_list_Test, batch_size=256, shuffle=False)
print(train_loader)
for step, data in enumerate(train_loader):
    print(data)
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data.batch)
    print()
    break

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_dim,hidden_channels,out_dim):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(in_dim, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, out_dim)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)
       

        # 2. Readout layer
        unique_batches = torch.unique(batch,sorted=True)
        final_weights = []
        for b in unique_batches:
            mask = (batch == b)
            batch_weights = batch[mask]
            for i,value in enumerate(batch[mask]):
                if i==0:
                    final_weights.append(node_weight)
                else:
                    final_weights.append((1-node_weight)/(len(batch_weights)-1))
                if len(batch_weights)==1:
                    print("Problem Here")

        final_weights = torch.tensor(final_weights)
        #print(final_weights)
        sum_weighted = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        sum_weighted.index_add_(0, batch, x * final_weights.view(-1, 1))

        x = sum_weighted  # [batch_size, hidden_channels]

        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.2):
        super().__init__()
        torch.manual_seed(12345)
        self.dropout = dropout
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        self.lin = Linear(hidden_dim, out_dim)
        
    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings 
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)
        # 2. Readout layer
        unique_batches = torch.unique(batch,sorted=True)
        final_weights = []
        for b in unique_batches:
            mask = (batch == b)
            batch_weights = batch[mask]
            for i,value in enumerate(batch[mask]):
                if i==0:
                    final_weights.append(node_weight)
                else:
                    final_weights.append((1-node_weight)/(len(batch_weights)-1))
                if len(batch_weights)==1:
                    print("Problem Here")

        final_weights = torch.tensor(final_weights)
        sum_weighted = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        sum_weighted.index_add_(0, batch, x * final_weights.view(-1, 1))

        x = sum_weighted  # [batch_size, hidden_channels]
        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

In [ ]:
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super().__init__()
        torch.manual_seed(12345)
        self.dropout = dropout
        self.conv1 = GATConv(in_dim, hidden_dim)
        self.conv2 = GATConv(hidden_dim, hidden_dim)
        self.conv3 = GATConv(hidden_dim, hidden_dim)
        self.lin = Linear(hidden_dim, out_dim)
        
    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings 
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)
        # 2. Readout layer
        unique_batches = torch.unique(batch,sorted=True)
        final_weights = []
        for b in unique_batches:
            mask = (batch == b)
            batch_weights = batch[mask]
            for i,value in enumerate(batch[mask]):
                if i==0:
                    final_weights.append(node_weight)
                else:
                    final_weights.append((1-node_weight)/(len(batch_weights)-1))
                if len(batch_weights)==1:
                    print("Problem Here")

        final_weights = torch.tensor(final_weights)
        sum_weighted = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        sum_weighted.index_add_(0, batch, x * final_weights.view(-1, 1))

        x = sum_weighted  # [batch_size, hidden_channels]
        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

### Training GNN and choosing best lambda value using optuna

In [ ]:
def objective(trial):
    weight_param=trial.suggest_float("weight_param",0.5,1.0)
    best_valid_loss= float('inf')
    model = GCN(384,64,2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()

    print("Weight : " ,weight_param)
    for epoch in range(1,11):
        #train(model_1,file,weight_param)
        model.train()
        for idx,data in enumerate(train_loader):  # Iterate in batches over the training dataset.
            out = model(data.x, data.edge_index, data.batch,weight_param)
            #out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass.
            loss = criterion(out, data.y)  # Compute the loss.

            if idx%50==0:
                print(idx,loss.item())

            loss.backward()  # Derive gradients.
            optimizer.step()  # Update parameters based on gradients.
            optimizer.zero_grad()  # Clear gradients.

        model.eval()

        total_loss = 0
        correct = 0
        total_samples = 0

        with torch.no_grad():  # No need to compute gradients during evaluation
            for data in valid_loader:  # Iterate in batches over the dataset
                out = model(data.x, data.edge_index, data.batch,weight_param)
                pred = out.argmax(dim=1)  # Get predicted classes
                correct += (pred == data.y).sum().item()  # Count correct predictions
                loss = criterion(out, data.y)  # Compute loss
                total_loss += loss.item() * data.num_graphs  # Accumulate loss (weighted by batch size)
                total_samples += data.num_graphs  # Accumulate total number of samples

        valid_loss = total_loss / total_samples
        valid_accuracy = correct / total_samples

        print("valid_loss ", valid_loss)

        model.eval()

        total_loss = 0
        correct = 0
        total_samples = 0

        with torch.no_grad():  # No need to compute gradients during evaluation
            for data in test_loader:  # Iterate in batches over the dataset
                out = model(data.x, data.edge_index, data.batch,weight_param)
                pred = out.argmax(dim=1)  # Get predicted classes
                correct += (pred == data.y).sum().item()  # Count correct predictions
                loss = criterion(out, data.y)  # Compute loss
                total_loss += loss.item() * data.num_graphs  # Accumulate loss (weighted by batch size)
                total_samples += data.num_graphs  # Accumulate total number of samples

        test_loss = total_loss / total_samples
        test_accuracy = correct / total_samples
        print("Test_loss ", test_loss)
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model, f"model_{weight_param}.pt")
            print(f"Model saved with valid loss: {valid_loss:.4f}")
        print(f'Epoch: {epoch:03d},  Test Acc: {test_accuracy:.4f}')

    return best_valid_loss

study=optuna.create_study(study_name='gcn_optuna',
    storage=storage_url,
    direction="minimize",
    load_if_exists=True)
study.optimize(objective,n_trials=50)
print('Best trial:')
trial = study.best_trial
print('  Value: ', trial.value)
print('  Params: ')
for key, value in trial.params.items():
    print(f'    {key}: {value}')